# 06 · Decision Framework

The build-and-debug guide, as a consolidated read-out of everything measured.

**Rules of thumb**
1. **Measure the four parts separately.** Retrieval recall first — it is the ceiling.
2. **Spend effort on chunking and retrieval before the model.** That is where the quality
   is; the model is the least impactful, most expensive knob.
3. **Choose the chunker to match the documents** (recursive/semantic for structured text,
   sentence-aware as a safe default); never accept the fixed-size default blindly.
4. **Add an abstention path + a groundedness check** — a RAG system that cannot say "I do
   not know" will hallucinate with citations.
5. **Know when RAG is the right tool at all:** facts that live in documents and change
   often → RAG; style/format → fine-tune (`lora_101`); stable/simple → just prompt.


In [1]:
import os, sys, json
from pathlib import Path

# Run from the repo root so outputs/ and data/ resolve no matter where the notebook starts.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import support
from src.config import load_settings
from src.datasets.corpus import build_corpus
from src.pipeline.embed import build_embedder

settings = load_settings()
corpus = build_corpus()
embedder = build_embedder(settings.embed)
generator, mode = support.select_generator(corpus, settings)
print("generator mode:", mode)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

generator mode: ollama:qwen2.5:0.5b


### Consolidated summary — recomputed from the pipeline

In [2]:
from src.datasets.corpus import chunk_corpus
from src.pipeline.chunk import make_chunk_fn
from src.pipeline.retrieve import build_retriever
from src.pipeline.rag import RagPipeline
from src.evaluation.ceiling import evaluate_ceiling
from src.evaluation.answer_metrics import abstention_rate

cf = make_chunk_fn(settings.chunk.strategy, size=settings.chunk.size,
                   overlap=settings.chunk.overlap, encoder=embedder)
chunks, chunk_doc_ids = chunk_corpus(corpus, cf)
retriever = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)

rag = RagPipeline(retriever=retriever, generator=generator, settings=settings, abstain=True)
rag.index(chunks)
ceiling = evaluate_ceiling(rag, corpus, chunk_doc_ids)


def abstention_on_absence(abstain):
    pipe = RagPipeline(retriever=build_retriever(settings.retrieve.method, embedder,
                       settings.retrieve.hybrid_alpha), generator=generator,
                       settings=settings, abstain=abstain)
    pipe.index(chunks)
    return abstention_rate([pipe.answer(q.text).answer
                            for q in corpus.questions_by_kind("absence")])


summary = {
    "mode": mode,
    "retrieval_recall": round(ceiling.retrieval_recall, 3),
    "oracle_accuracy": round(ceiling.oracle_accuracy, 3),
    "retrieved_accuracy": round(ceiling.retrieved_accuracy, 3),
    "abstention_naive": abstention_on_absence(False),
    "abstention_grounded": abstention_on_absence(True),
}
summary


{'mode': 'ollama:qwen2.5:0.5b',
 'retrieval_recall': 1.0,
 'oracle_accuracy': 1.0,
 'retrieved_accuracy': 1.0,
 'abstention_naive': 0.5,
 'abstention_grounded': 0.5}

### When to use RAG vs fine-tune vs prompt

In [3]:
decision = pd.DataFrame([
    {"need": "Facts that live in documents and change often", "use": "RAG"},
    {"need": "Style, tone or output format", "use": "Fine-tune (lora_101)"},
    {"need": "Stable, simple, general knowledge", "use": "Just prompt"},
])
decision


,need,use
0,Facts that live in documents and change often,RAG
1,"Style, tone or output format",Fine-tune (lora_101)
2,"Stable, simple, general knowledge",Just prompt


### Figure → PNG, summary → JSON

In [4]:
from src.visualisation import bar_chart

fig = bar_chart(["retrieval recall", "oracle acc", "retrieved acc"],
                [summary["retrieval_recall"], summary["oracle_accuracy"], summary["retrieved_accuracy"]],
                title="The ceiling in one picture", ylabel="score", ylim=(0, 1.05))
p = support.save_fig(fig, "06_summary")
support.save_json({**summary, "decision_table": decision.to_dict(orient="records")}, "06_framework")
print("saved", p)
fig


saved outputs/figures/06_summary.png


<Figure size 700x400 with 1 Axes>